# Profile-RAG Principle Model — Notebook

Predicts Big Five traits using `principle_model.predict` against a **profile-based**
vector DB built from `rag.profiler` outputs.

Pipeline:
1. Load training profiles from `data/profile_db/essays/profile_store.jsonl`
   (built by `notebook/data_process/profile_build.ipynb`).
2. Build a per-trait FAISS index keyed off the *trait-sliced* profile of each essay.
3. At inference, profile each test essay (label-blind) and retrieve top-k profile-similar
   neighbours from the trait-specific index.
4. Render the retrieved neighbours' profile slices as the few-shot exemplars in the
   `principle_model` prompt.
5. Run `predict` + `evaluate`.

**Prompt modes supported here:** `rag_zeroshot`, `rag_oneshot`,
`cot_rag_zeroshot`, `cot_rag_oneshot` (the four RAG modes in `principle_model`).

In [1]:
from pathlib import Path
import sys, os, json, shutil, time
from typing import Dict, List

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "principle_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.profiler.store import ProfileStore
from rag.profiler.runner import build_profiles
from rag.profiler.prompts import (
    FACETS,
    TRAIT_FACETS_FOR_INFERENCE,
    TRAIT_LING_FOR_INFERENCE,
    slice_profile_for_trait,
)
from rag.faiss_index import FAISSIndex
import rag.embedder as _embedder
import rag.retriever as _retriever_mod

from principle_model.predict import predict
from principle_model.evaluate import evaluate

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [2]:
# --- Paths ----------------------------------------------------------------
train_csv         = str(project_root / "data/split/essays/train.csv")
test_csv          = str(project_root / "data/split/essays/test50.csv")
principles_dir    = str(project_root / "data/principles")

train_profile_db  = str(project_root / "data/profile_db/essays")             # built by profile_build.ipynb
test_profile_db   = str(project_root / "data/profile_db/essays_test")        # generated below if missing
vector_db_dir     = str(project_root / "data/vector_db/essays_profile")      # FAISS index built from profiles

res_dir           = str(project_root / "result")
log_dir           = str(project_root / "log")

# --- Profiler model (used only if test profiles need to be generated) ----
profiler_model    = "gpt-4o-mini"

# --- Predictor model & prompt --------------------------------------------
model_name        = "gpt-4o-mini"      # the classifier; same as your baseline rag_oneshot run
max_new_tokens    = 512
temperature       = 0.0
prompt_mode       = "rag_oneshot"      # rag_zeroshot | rag_oneshot | cot_rag_zeroshot | cot_rag_oneshot
top_k             = 3

# --- DB rebuild knobs ----------------------------------------------------
force_rebuild_db  = False              # set True to wipe vector_db_dir and rebuild from scratch

test_df = pd.read_csv(test_csv)
print(f"Test  : {len(test_df):>4} rows  | mode={prompt_mode}  | top_k={top_k}")

Test  :   50 rows  | mode=rag_oneshot  | top_k=3


## Step 1 — Load training profiles

In [3]:
train_store = ProfileStore(str(Path(train_profile_db) / "profile_store.jsonl"))
train_store.load()
train_entries = [e for e in train_store.get_all() if e.get("valid")]
print(f"Loaded {len(train_entries)} valid training profiles from {train_profile_db}")
if not train_entries:
    raise RuntimeError(
        f"No training profiles found at {train_profile_db}. "
        "Run notebook/data_process/profile_build.ipynb first."
    )

Loaded 1974 valid training profiles from F:\std\GR\code\model_x_ocean\data\profile_db\essays


## Step 2 — Profile the test set (label-blind)

Test profiles MUST be generated **without exposing labels** (`use_labels=False`)
to avoid leakage: at inference time the model would not know the ground truth.

In [4]:
test_store_path = Path(test_profile_db) / "profile_store.jsonl"
test_store = ProfileStore(str(test_store_path))
test_store.load()
needed = len(test_df) - sum(
    1 for i in range(len(test_df)) if test_store.has(f"user_{i}") and test_store.get(f"user_{i}").get("valid")
)
print(f"Test profiles already in store: {len(test_store)}; missing: {needed}")

if needed > 0:
    test_store = build_profiles(
        data       = test_df,
        output_dir = test_profile_db,
        model_name = profiler_model,
        log_dir    = str(Path(log_dir) / "profiler_test"),
        use_labels = False,            # IMPORTANT: label-blind for test
    )
test_entries_by_idx = {
    int(e["user_id"].split("_")[1]): e for e in test_store.get_all() if e.get("valid")
}
print(f"Test profiles ready: {len(test_entries_by_idx)}")

Test profiles already in store: 50; missing: 0
Test profiles ready: 50


## Step 3 — Build the profile-based vector DB

Layout (legacy `FeatureRAGRetriever` format):
* `vectors.faiss` — embeddings of the rendered profiles.
* `vectors_meta.jsonl` — per-vector metadata: user_id, trait_labels, the full parsed
  profile under `features["profile"]`, and the embedded text under
  `posts_raw` (so retrieval is profile-vs-profile, but the raw essay still travels
  in case downstream code needs it).
* `feature_store.jsonl` — duplicate of the per-user profile, in the format
  `FeatureStore` expects.

We embed **the full rendered profile** (`raw` field). At retrieval time we will
render the *trait-sliced* version and pass that as the query — see the
`ProfileRAGRetriever` adapter below.

In [5]:
def render_full_profile_text(entry: Dict) -> str:
    """Stable, deterministic rendering of a profile for embedding.
    Uses the ``raw`` field if present, otherwise reconstructs from facets/linguistic.
    """
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)

In [6]:
# Force the embedder to use the fine-tuned SBERT (or whatever default) — same model
# is used to embed both train profiles and the query profile at inference.
from rag.embedder import get_embedding

TRAIT_NAMES = ("Openness to Experience","Conscientiousness","Extraversion","Agreeableness","Neuroticism")
TRAIT_CODES = {
    "Openness to Experience": "cOPN",
    "Conscientiousness":      "cCON",
    "Extraversion":           "cEXT",
    "Agreeableness":          "cAGR",
    "Neuroticism":            "cNEU",
}

def build_profile_vector_db(entries: List[Dict], output_dir: str, force: bool = False) -> None:
    os.makedirs(output_dir, exist_ok=True)
    index_path = os.path.join(output_dir, "vectors.faiss")
    meta_path  = os.path.join(output_dir, "vectors_meta.jsonl")
    fs_path    = os.path.join(output_dir, "feature_store.jsonl")

    if not force and os.path.exists(index_path) and os.path.exists(meta_path):
        print(f"[profile-db] Already built at {output_dir}; skipping.")
        return

    if force:
        for p in (index_path, meta_path, fs_path):
            if os.path.exists(p):
                os.remove(p)

    print(f"[profile-db] Embedding {len(entries)} profiles -> {output_dir}")
    texts_to_embed = [render_full_profile_text(e) for e in entries]
    embeddings = get_embedding(texts_to_embed)
    embeddings = np.array(embeddings, dtype="float32")

    meta, fs_lines = [], []
    for e, embedded_text in zip(entries, texts_to_embed):
        labels = e.get("trait_labels", {})
        # Convert trait label codes (cOPN/...) to full names used by the retriever
        rag_labels = {}
        for code, name in [("cOPN", "Openness to Experience"), ("cCON", "Conscientiousness"),
                           ("cEXT", "Extraversion"), ("cAGR", "Agreeableness"), ("cNEU", "Neuroticism")]:
            if code in labels and labels[code] in ("high", "low"):
                rag_labels[name] = labels[code]
        # The full parsed profile (kept under features["profile"]) lets the
        # retriever adapter render trait-specific slices at inference.
        features = {"profile": {"facets": e.get("facets", {}), "linguistic": e.get("linguistic", {})}}
        meta.append({
            "user_id":      e["user_id"],
            "posts_raw":    embedded_text,        # the embedded profile text (NOT the raw essay)
            "trait_labels": rag_labels,
            "features":     features,
        })
        fs_lines.append({
            "user_id":      e["user_id"],
            "trait_labels": rag_labels,
            "features":     features,
        })

    idx = FAISSIndex(dimension=embeddings.shape[1])
    idx.build(embeddings, meta)
    idx.save(index_path, meta_path)
    with open(fs_path, "w", encoding="utf-8") as f:
        for line in fs_lines:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")
    print(f"[profile-db] Saved index ({embeddings.shape[0]} vectors, dim={embeddings.shape[1]}).")

build_profile_vector_db(train_entries, vector_db_dir, force=force_rebuild_db)

[profile-db] Embedding 1974 profiles -> F:\std\GR\code\model_x_ocean\data\vector_db\essays_profile
[embedder] Loading embedding model: F:\std\GR\code\model_x_ocean\models\sbert_essays_finetuned


<All keys matched successfully>


KeyboardInterrupt: 

## Step 4 — Profile-aware retriever adapter

Subclasses `FeatureRAGRetriever` to:
* embed the **test essay's profile** (not the raw essay) as the query,
* render retrieved exemplars as **trait-sliced profiles + label**
  (instead of raw text + LIWC features).

The adapter is monkey-patched onto `rag.retriever.FeatureRAGRetriever` so that
`principle_model.predict` picks it up without any change to the predictor code.
We disable the fine-tuned-artifacts path explicitly so the legacy `db_dir`
(our profile-based DB) is used.

In [ ]:
from rag.retriever import FeatureRAGRetriever

class ProfileRAGRetriever(FeatureRAGRetriever):
    """Retriever that:
      - embeds the QUERY essay's parsed profile (not the raw essay),
      - returns trait-sliced profile excerpts as few-shot exemplars.
    """

    def __init__(self, db_dir: str, test_profiles_by_idx: Dict[int, Dict], test_df: pd.DataFrame):
        # Force legacy path (our profile-based DB lives there)
        super().__init__(db_dir=db_dir)
        # Override the auto-detection of the fine-tuned artifacts dir
        self._use_finetuned = False
        self._test_profiles_by_idx = test_profiles_by_idx
        # Index test essays by their text so build_similar_context can find the profile
        self._text_to_idx = {str(t): i for i, t in enumerate(test_df["text"].tolist())}

    def _embed_query_profile(self, query_text: str):
        idx = self._text_to_idx.get(str(query_text))
        if idx is None or idx not in self._test_profiles_by_idx:
            # Fallback: embed the raw text. Better than crashing, but signals a missing profile.
            print(f"  [retriever] WARN: no test profile found for query (idx={idx}); falling back to raw text.")
            return self._embed(query_text)
        profile_text = render_full_profile_text(self._test_profiles_by_idx[idx])
        return np.array(self._embed(profile_text), dtype="float32")

    def build_similar_context(self, posts: str, trait: str, top_k: int = 3) -> str:
        # `trait` is the full RAG name ("Openness to Experience", ...)
        trait_code = TRAIT_CODES.get(trait)
        if trait_code is None:
            return super().build_similar_context(posts=posts, trait=trait, top_k=top_k)

        query_emb = self._embed_query_profile(posts)
        all_results = self._search(query_emb, top_k * 4)

        blocks, seen = [], 0
        for r in all_results:
            if trait not in r.get("trait_labels", {}):
                continue
            label = r["trait_labels"][trait]
            features = r.get("features", {}) or {}
            profile = features.get("profile") or {}
            slice_text = slice_profile_for_trait(profile, trait_code) if profile else ""
            if not slice_text.strip():
                slice_text = "  (no profile slice available)"
            blocks.append(
                f"[Similar Profile {seen+1}] (label: {label})\n{slice_text}"
            )
            seen += 1
            if seen >= top_k:
                break
        return "\n\n".join(blocks)

# Monkey-patch so principle_model.predict instantiates the profile-aware retriever
_test_profiles_by_idx_local = test_entries_by_idx
_test_df_local = test_df

_OriginalRetriever = _retriever_mod.FeatureRAGRetriever

def _RetrieverFactory(db_dir=None):
    return ProfileRAGRetriever(
        db_dir=db_dir or vector_db_dir,
        test_profiles_by_idx=_test_profiles_by_idx_local,
        test_df=_test_df_local,
    )

_retriever_mod.FeatureRAGRetriever = _RetrieverFactory
# Also disable the auto-fine-tuned-dir promotion so predict() uses our db_dir
_retriever_mod.FINETUNED_ARTIFACTS_DIR = None
print("[adapter] ProfileRAGRetriever installed.")

## Step 5 — Sanity check: render one few-shot context

Confirms the retriever round-trips: query the first test essay's profile and
look at what would actually be shown to the classifier for Conscientiousness.

In [ ]:
_smoke = _RetrieverFactory(db_dir=vector_db_dir)
_query_text = test_df.iloc[0]["text"]
for trait_full in TRAIT_NAMES:
    print(f"\n=== {trait_full} ===")
    print(_smoke.build_similar_context(posts=_query_text, trait=trait_full, top_k=top_k))

## Step 6 — Run prediction

In [ ]:
run_id, run_time, prediction_csv = predict(
    text_df        = test_df,
    model_name     = model_name,
    log_dir        = log_dir,
    prompt_mode    = prompt_mode,
    max_new_tokens = max_new_tokens,
    res_dir        = res_dir,
    temperature    = temperature,
    principles_dir = principles_dir,
    top_k          = top_k,
    vector_db_dir  = vector_db_dir,    # <- our profile-based DB
)

print(f"\nDone in {run_time:.1f}s")
print(f"Predictions saved to: {prediction_csv}")

## Step 7 — Evaluate

In [ ]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = run_time,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

print("Summary CSV:", evaluation["summary_csv"])
print(f"Failed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")
summary_df = pd.read_csv(evaluation["summary_csv"])
display(summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))

## Step 8 — Restore the original retriever (cleanup)

If you keep the kernel alive, restore the original retriever class so subsequent
cells in other notebooks aren't affected by the monkey-patch.

In [ ]:
_retriever_mod.FeatureRAGRetriever = _OriginalRetriever
print("[adapter] Original FeatureRAGRetriever restored.")